# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Raja-saab/Flyrank1/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

I first inspect the distributions of the main decision-time signals used by the baseline: `days_since_last_update`, `impressions_90d`, `avg_position`, and `ctr`.

These fields are expected to be unevenly distributed because a small number of pages can have much larger traffic or much older update dates than the typical page. I will use the observed distributions to choose sensible buckets rather than assuming the fields are normally distributed.

In [5]:
from pathlib import Path
import numpy as np
import pandas as pd

# Find repository
candidates = [
    Path("/content/Flyrank1"),
    Path.cwd(),
]

REPO_ROOT = None

for candidate in candidates:
    if (candidate / "data/raw/content_refresh_anonymized.csv").exists():
        REPO_ROOT = candidate
        break

if REPO_ROOT is None:
    matches = list(
        Path("/content").glob(
            "**/data/raw/content_refresh_anonymized.csv"
        )
    )
    if matches:
        REPO_ROOT = matches[0].parents[2]

if REPO_ROOT is None:
    raise FileNotFoundError(
        "Could not find data/raw/content_refresh_anonymized.csv"
    )

DATA_PATH = REPO_ROOT / "data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

print("Rows:", len(df))
print("Columns:", len(df.columns))

required = [
    "content_id",
    "impressions_90d",
    "days_since_last_update",
    "avg_position",
    "ctr",
    "trend_direction",
    "trend_pct",
]

missing = [c for c in required if c not in df.columns]

if missing:
    raise ValueError(f"Missing columns: {missing}")

numeric_cols = [
    "impressions_90d",
    "days_since_last_update",
    "avg_position",
    "ctr",
    "trend_pct",
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["trend_direction"] = (
    df["trend_direction"]
    .fillna("unknown")
    .astype(str)
)

audit_df = df[
    (df["impressions_90d"] > 0)
    & (df["days_since_last_update"] >= 0)
].copy()

audit_df = audit_df.drop_duplicates("content_id")

signal_cols = [
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
]

print("\nSignal summary:")
display(
    audit_df[signal_cols].describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    ).T
)

print("\nMissingness:")
display(
    audit_df[signal_cols]
    .isna()
    .mean()
    .mul(100)
    .round(2)
    .rename("missing_pct")
    .to_frame()
)

Rows: 30000
Columns: 44

Signal summary:


,count,mean,std,min,25%,50%,75%,90%,95%,99%,max
days_since_last_update,30000.0,46.098300,42.078709,1.0,20.0,20.00,104.00,104.00,104.00,106.000,373.0
impressions_90d,30000.0,5200.366300,16838.019547,1.0,81.0,731.00,3615.25,12136.40,22996.50,73505.830,517715.0
avg_position,30000.0,16.342380,15.216790,0.0,6.2,10.80,22.30,36.80,48.20,69.901,245.0
ctr,30000.0,0.510733,3.279162,0.0,0.0,0.07,0.29,0.65,1.09,8.330,100.0



Missingness:


,missing_pct
days_since_last_update,0.0
impressions_90d,0.0
avg_position,0.0
ctr,0.0


## 2. Signal test #1 / #2 / #3

I test three decision-time signals:

1. `days_since_last_update` — staleness
2. `impressions_90d` — search visibility
3. `avg_position` — search ranking opportunity

For each signal I compare buckets using the observed declining rate. The label is used only for this audit; it is not used as an input to the baseline score.

The verdicts are based on the observed direction:

- **CONFIRMED** — the observed pattern supports the assumption
- **OPPOSITE** — the observed pattern runs against the assumption
- **MIXED** — the relationship is inconsistent
- **FALSE** — there is little/no evidence for the assumption

In [6]:
# -------------------------------------------------------
# Signal #1: Staleness
# -------------------------------------------------------

audit_df["staleness_bucket"] = pd.cut(
    audit_df["days_since_last_update"],
    bins=[-1, 30, 90, 180, 365, np.inf],
    labels=[
        "0-30d",
        "31-90d",
        "91-180d",
        "181-365d",
        "365+d",
    ],
)

staleness_test = (
    audit_df
    .groupby("staleness_bucket", observed=False)
    .agg(
        n=("content_id", "size"),
        median_impressions=("impressions_90d", "median"),
        declining_rate=(
            "trend_direction",
            lambda x: (
                x.str.lower() == "down"
            ).mean()
        ),
    )
    .reset_index()
)

print("SIGNAL #1 — STALENESS")
display(staleness_test)


# -------------------------------------------------------
# Signal #2: Search visibility
# -------------------------------------------------------

audit_df["visibility_bucket"] = pd.cut(
    audit_df["impressions_90d"],
    bins=[0, 100, 500, 3000, 30000, np.inf],
    labels=[
        "1-100",
        "101-500",
        "501-3k",
        "3k-30k",
        "30k+",
    ],
)

visibility_test = (
    audit_df
    .groupby("visibility_bucket", observed=False)
    .agg(
        n=("content_id", "size"),
        median_staleness=(
            "days_since_last_update",
            "median"
        ),
        declining_rate=(
            "trend_direction",
            lambda x: (
                x.str.lower() == "down"
            ).mean()
        ),
    )
    .reset_index()
)

print("\nSIGNAL #2 — SEARCH VISIBILITY")
display(visibility_test)


# -------------------------------------------------------
# Signal #3: Average position
# -------------------------------------------------------

audit_df["position_bucket"] = pd.cut(
    audit_df["avg_position"],
    bins=[-np.inf, 3, 10, 20, 50, np.inf],
    labels=[
        "Top 3",
        "4-10",
        "11-20",
        "21-50",
        "50+",
    ],
)

position_test = (
    audit_df
    .groupby("position_bucket", observed=False)
    .agg(
        n=("content_id", "size"),
        median_impressions=(
            "impressions_90d",
            "median"
        ),
        declining_rate=(
            "trend_direction",
            lambda x: (
                x.str.lower() == "down"
            ).mean()
        ),
    )
    .reset_index()
)

print("\nSIGNAL #3 — AVERAGE POSITION")
display(position_test)

SIGNAL #1 — STALENESS


,staleness_bucket,n,median_impressions,declining_rate
0,0-30d,20480,470.0,0.511377
1,31-90d,175,510.0,0.588571
2,91-180d,9171,1692.0,0.611057
3,181-365d,169,16.0,0.467456
4,365+d,5,2.0,0.600000



SIGNAL #2 — SEARCH VISIBILITY


,visibility_bucket,n,median_staleness,declining_rate
0,1-100,8006,20.0,0.389208
1,101-500,5279,22.0,0.604281
2,501-3k,8432,22.0,0.620849
3,3k-30k,7205,22.0,0.586121
4,30k+,1078,25.0,0.461967



SIGNAL #3 — AVERAGE POSITION


,position_bucket,n,median_impressions,declining_rate
0,Top 3,2346,3.0,0.245524
1,4-10,11842,1184.0,0.569414
2,11-20,7273,870.0,0.609515
3,21-50,7225,807.0,0.561799
4,50+,1314,219.5,0.343227


## 3. The flag-linked test

The flag-linked signal is `days_since_last_update`.

This is directly connected to FlyRank's refresh/staleness logic: a page that has not been updated for a long time is a candidate for refresh review.

I test whether the observed declining rate changes across staleness buckets. The purpose is not to prove causation, but to check whether the signal is directionally useful enough to support a rule.

In [7]:
# Flag-linked signal: staleness / refresh logic

flag_test = (
    audit_df
    .groupby("staleness_bucket", observed=False)
    .agg(
        n=("content_id", "size"),
        declining_rate=(
            "trend_direction",
            lambda x: (
                x.str.lower() == "down"
            ).mean()
        ),
        median_days_since_update=(
            "days_since_last_update",
            "median"
        ),
    )
    .reset_index()
)

display(flag_test)

print("\nFlag-linked signal: days_since_last_update")
print(
    "This signal is decision-time information and is directly related "
    "to the refresh/staleness flag logic."
)

,staleness_bucket,n,declining_rate,median_days_since_update
0,0-30d,20480,0.511377,20.0
1,31-90d,175,0.588571,41.0
2,91-180d,9171,0.611057,104.0
3,181-365d,169,0.467456,211.0
4,365+d,5,0.600000,373.0



Flag-linked signal: days_since_last_update
This signal is decision-time information and is directly related to the refresh/staleness flag logic.


## 4. What this means in practice

The audit checks whether the signals behind the refresh rule show useful directional patterns in the observed data. Staleness is particularly useful because it is directly actionable: a content team can review an older page for a possible refresh.

The signals should be treated as decision-support rather than proof that a page will decline or that refreshing it will definitely improve performance. A mixed signal is still useful because it tells us where the baseline needs caution.

In [8]:
print(
    "Signal audit complete. "
    "The tables above show the observed evidence for each signal."
)

Signal audit complete. The tables above show the observed evidence for each signal.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.